<p style="font-size:16pt; font-weight:bold; color:red; padding-bottom:20px; float:right">Please rename this file before editing!</p>

# Session 10 — A Worked Example: One Person at the Help Desk, or Two?

Notebook 01 simulated dice and doors, whose answers are known. This notebook
simulates something with no formula you could look up, and carries it through
to a recommendation.

**The situation.** A campus IT help desk is open eight hours a day. Students
walk up at random, about one every four minutes. Helping one takes about three
and a half minutes, sometimes one, sometimes ten. One person staffs the desk.
Students complain about the wait.

**The question:** *How long do students wait, and is a second person at the
desk worth it?*

On paper the desk keeps up: a student every 4 minutes, 3.5 minutes each. The
person at the desk is even idle an eighth of the time. So why is there a queue
at all? Because neither number is constant, and a queue is what randomness
looks like when it piles up.

| Step | What happens |
|---|---|
| 1 | Write the model down: what is fixed, what is random |
| 2 | One day, as arithmetic on two arrays |
| 3 | Check the tool on cases with a known answer |
| 4 | One random day, looked at closely |
| 5 | Many days: an estimate with an interval |
| 6 | One person or two, compared on the same days |
| 7 | What if the desk gets busier? |
| 8 | The recommendation, and what the model leaves out |

> **Run this on ARC, with the kernel set to Conda Python 3.12.** No data file
> is needed. The whole notebook takes about ten seconds.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.options.display.float_format = "{:,.2f}".format

SEED = 2026

---

## Step 1 — The model

A model is the situation reduced to the few rules that matter, written down so
that someone else can disagree with them.

| | Rule | Value |
|---|---|---|
| The desk is open | a fixed time | 480 minutes |
| Gap between two arrivals | random, exponential | 4 minutes on average |
| Time to help one student | random, exponential | 3.5 minutes on average |
| Queue | first come, first served; nobody gives up | |
| Staff | each helps one student at a time | 1, or 2 |

The **exponential** distribution is the usual model for "time until the next
thing happens" when things happen independently: many short gaps, a few long
ones.

In [ ]:
OPEN_MINUTES = 480
MEAN_GAP = 4.0
MEAN_SERVICE = 3.5

rng = np.random.default_rng(SEED)
gaps = rng.exponential(MEAN_GAP, size=10_000)

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.hist(gaps, bins=50, color="steelblue", edgecolor="white")
ax.axvline(gaps.mean(), color="firebrick", linestyle="--", label="mean {:.1f}".format(gaps.mean()))
ax.axvline(np.median(gaps), color="black", linestyle=":", label="median {:.1f}".format(np.median(gaps)))
ax.set_xlabel("minutes between two arrivals")
ax.set_ylabel("gaps (of 10,000)")
ax.set_title("Arrivals come in bursts: most gaps are short")
ax.legend()
fig.tight_layout()

Skewed, as Session 9 would predict from a mean well above the median. Half of
all gaps are under three minutes. Students do not arrive four minutes apart;
they arrive in clumps with lulls between, and the clumps are where queues
come from.

**The randomness goes into one function, and nothing else is random.** It
returns two arrays: when each student arrives, and how long each will need.

In [ ]:
def draw_day(rng, mean_gap=MEAN_GAP):
    """Draw one day: arrival times and service times, in minutes."""
    # Far more gaps than a day can hold, then keep the arrivals before closing.
    gaps = rng.exponential(mean_gap, size=int(3 * OPEN_MINUTES / mean_gap) + 50)
    arrivals = np.cumsum(gaps)
    arrivals = arrivals[arrivals < OPEN_MINUTES]
    services = rng.exponential(MEAN_SERVICE, size=len(arrivals))
    return arrivals, services


arrivals, services = draw_day(np.random.default_rng(SEED))
print(len(arrivals), "students today")
print("first arrivals:", np.round(arrivals[:6], 1))
print("their services:", np.round(services[:6], 1))

---

## Step 2 — One day, as arithmetic

Given the two arrays, a day is no longer random. Each staff member has a time
at which they are next free. A student is helped by whoever is free first:

```text
start = the later of (the student's arrival, the moment that staff member is free)
wait  = start − arrival
that staff member is free again at start + service time
```

This has to be a loop: each student's wait depends on the students before.
Not everything vectorizes.

In [ ]:
def waits(arrivals, services, staff=1):
    """Return each student's wait in minutes.  Nothing here is random."""
    free_at = np.zeros(staff)                   # when each staff member is next free
    waited = np.empty(len(arrivals))
    for number in range(len(arrivals)):
        who = np.argmin(free_at)                # the one who is free first
        start = max(arrivals[number], free_at[who])
        waited[number] = start - arrivals[number]
        free_at[who] = start + services[number]
    return waited

---

## Step 3 — Check the tool on known answers

Before believing the simulation on a random day, try it where the answer is
obvious. Because `waits` takes its inputs as arguments, any day can be handed
to it, including days that are not random at all.

**Case 1.** A student every 4 minutes, each needing exactly 3. Nobody waits.

In [ ]:
steady_arrivals = np.arange(0, 40, 4.0)
print(waits(steady_arrivals, np.full(10, 3.0)))

**Case 2.** A student every 4 minutes, each needing exactly 5. The desk falls
one minute further behind with every student: waits of 0, 1, 2, 3, ...

In [ ]:
print(waits(steady_arrivals, np.full(10, 5.0)))

**Case 3.** The same overloaded day with two staff. Two people handle a
student every 4 minutes with ease.

In [ ]:
print(waits(steady_arrivals, np.full(10, 5.0), staff=2))

**Case 4.** Three students arrive at the same moment, each needing 2 minutes.
With one person they wait 0, 2 and 4. With two, 0, 0 and 2.

In [ ]:
together = np.array([10.0, 10.0, 10.0])
print(waits(together, np.full(3, 2.0), staff=1), waits(together, np.full(3, 2.0), staff=2))

Four cases, four answers known in advance, four matches. None of this proves
the function right. It makes the kinds of mistake that matter (an off-by-one
in the queue, the wrong staff member chosen) much less likely to be hiding.

This is the simulation's version of testing, and it was possible only because
the random part and the arithmetic part are separate functions.

**✏️ Try it:** Make up a fifth case you can work out by hand: two students, the second arriving while the first is still being helped. Predict the waits, then run it.

---

## Step 4 — One random day, looked at closely

In [ ]:
arrivals, services = draw_day(np.random.default_rng(SEED))
one_day = waits(arrivals, services)

print("students {}   mean wait {:.1f} min   median {:.1f}   longest {:.1f}".format(
    len(one_day), one_day.mean(), np.median(one_day), one_day.max()))
print("waited at all: {:.0%}   waited more than 10 minutes: {:.0%}".format(
    (one_day > 0).mean(), (one_day > 10).mean()))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.6))
ax.plot(arrivals / 60, one_day, color="steelblue")
ax.set_xlabel("hours since the desk opened")
ax.set_ylabel("wait of the student arriving then (minutes)")
ax.set_title("One day at the desk: quiet stretches, then pile-ups that take an hour to clear")
fig.tight_layout()

The wait is not a steady few minutes. Long stretches are near zero, and then a
burst of arrivals, or one student with a long problem, builds a queue that
everyone behind them inherits. A queue remembers.

Is this a typical day? One day cannot say. Here are four more seeds.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(13, 2.8), sharey=True)
for ax, seed in zip(axes, (4, 2, 6, 20)):
    day_arrivals, day_services = draw_day(np.random.default_rng(seed))
    day_waits = waits(day_arrivals, day_services)
    ax.plot(day_arrivals / 60, day_waits, color="steelblue")
    ax.set_title("seed {}: mean wait {:.1f} min".format(seed, day_waits.mean()))
    ax.set_xlabel("hours since opening")
axes[0].set_ylabel("wait (minutes)")
fig.tight_layout()

Same desk, same rules, and the mean wait runs from 5 minutes to 23.
Anyone who visited on one of these days would describe a different help desk.
A single run of a simulation is an anecdote.

---

## Step 5 — Many days

One trial is one day. Its outcome is that day's mean wait. Everything from
notebook 01 applies: one generator, made once, used for every day.

In [ ]:
def simulate_days(days, rng, staff=1, mean_gap=MEAN_GAP):
    """Return the mean wait of each of `days` simulated days."""
    daily = np.empty(days)
    for day in range(days):
        arrivals, services = draw_day(rng, mean_gap)
        daily[day] = waits(arrivals, services, staff).mean()
    return daily


def estimate(outcomes):
    """Mean, standard error and interval of the outcomes."""
    mean = outcomes.mean()
    se = outcomes.std(ddof=1) / np.sqrt(len(outcomes))
    return {"trials": len(outcomes), "mean": mean, "se": se,
            "low": mean - 1.96 * se, "high": mean + 1.96 * se}


DAYS = 2000
daily_one = simulate_days(DAYS, np.random.default_rng(SEED))
estimate(daily_one)

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 3.6))
ax.hist(daily_one, bins=40, color="steelblue", edgecolor="white")
ax.axvline(daily_one.mean(), color="firebrick", linestyle="--",
           label="mean {:.1f}".format(daily_one.mean()))
ax.axvline(np.median(daily_one), color="black", linestyle=":",
           label="median {:.1f}".format(np.median(daily_one)))
ax.set_xlabel("mean wait on a day (minutes)")
ax.set_ylabel("days (of {:,})".format(DAYS))
ax.set_title("Most days are bearable; some are terrible")
ax.legend()

print("days with a mean wait over 20 minutes: {:.1%}".format((daily_one > 20).mean()))
print("the worst day: {:.0f} minutes".format(daily_one.max()))
fig.tight_layout()

Two different things are now on the table, and they must not be confused.

- The **interval** from `estimate` says how well the simulation knows the
  *long-run average* wait. It is narrow, because 2,000 days were averaged.
- The **histogram** says how much *one day* varies. That spread is large, and
  no number of trials will shrink it: it is a fact about the desk, not about
  the simulation.

A student does not experience the long-run average. They experience a day.
The histogram, with its long tail of bad days, is the more useful picture for
whoever runs the desk, and "the average wait is such-and-such minutes" on its
own would hide it.

**✏️ Try it:** The outcome of a trial can be anything computed from a day. Change `simulate_days` (or write a second version) so that the outcome is the share of students who wait more than 10 minutes.

---

## Step 6 — One person or two, on the same days

The comparison that matters. Both staffings must face **the same students**:
the same arrivals and the same service times, day by day. Draw each day once
and compute the waits twice.

In [ ]:
def compare_staffing(days, rng, mean_gap=MEAN_GAP):
    """Mean wait with one and with two staff, on the same days."""
    one = np.empty(days)
    two = np.empty(days)
    for day in range(days):
        arrivals, services = draw_day(rng, mean_gap)      # drawn once
        one[day] = waits(arrivals, services, staff=1).mean()
        two[day] = waits(arrivals, services, staff=2).mean()
    return one, two


one, two = compare_staffing(DAYS, np.random.default_rng(SEED))

table = pd.DataFrame([estimate(one), estimate(two), estimate(one - two)],
                     index=["one person", "two people", "minutes saved per student"])
table

A second person does not halve the wait. It all but removes it. With two
people, a student who arrives during someone's ten-minute problem goes to the
other one, and the queue never gets the chance to build.

The last row is the estimate of the **difference**, made from the daily
differences. Its interval is nowhere near zero: this is not a close call, and
no amount of bad luck in the random numbers explains it.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 3.6))
edges = np.linspace(0, max(one.max(), two.max()), 50)
ax.hist(one, bins=edges, color="steelblue", alpha=0.7, label="one person")
ax.hist(two, bins=edges, color="darkorange", alpha=0.7, label="two people")
ax.set_xlabel("mean wait on a day (minutes)")
ax.set_ylabel("days (of {:,})".format(DAYS))
ax.set_title("With two people, the bad days are gone")
ax.legend()
fig.tight_layout()

---

## Step 7 — What if the desk gets busier?

The model's inputs are guesses. "One student every four minutes" came from
somebody's impression. A recommendation that holds only for exactly that
number is worth little, so vary it: this is a **sensitivity analysis**.

In [ ]:
rows = []
for mean_gap in (6.0, 5.0, 4.5, 4.0, 3.75, 3.5):
    one, two = compare_staffing(500, np.random.default_rng(SEED), mean_gap)
    rows.append({"minutes between arrivals": mean_gap,
                 "desk busy (share of time)": MEAN_SERVICE / mean_gap,
                 "wait, one person": one.mean(),
                 "wait, two people": two.mean()})
sensitivity = pd.DataFrame(rows)
sensitivity

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 3.8))
ax.plot(sensitivity["desk busy (share of time)"] * 100, sensitivity["wait, one person"],
        marker="o", color="steelblue", label="one person")
ax.plot(sensitivity["desk busy (share of time)"] * 100, sensitivity["wait, two people"],
        marker="s", color="darkorange", linestyle="--", label="two people")
ax.set_ylim(bottom=0)
ax.set_xlabel("share of the day one person would be busy (%)")
ax.set_ylabel("mean wait (minutes)")
ax.set_title("Near full use, a little more traffic means a lot more waiting")
ax.legend()
fig.tight_layout()

This is the finding that no back-of-envelope calculation gives. The wait does
not grow in step with the traffic. At 58 % use it is about five minutes. At 88 %,
where the desk is now, it is sixteen, and at 100 % twenty-eight: each step in
traffic costs more than the one before. A modest rise in arrivals, the first
week of term, say, adds ten minutes to everybody's wait.

With two people the curve stays near one minute across the whole range. The second person
is not there for the average day. They are insurance against the busy one.

---

## Step 8 — The recommendation, and what the model leaves out

What an honest statement of the result includes:

> A simulation of the help desk (arrivals every 4 minutes and service of 3.5
> minutes on average, both exponential; 2,000 simulated days; seed 2026) gives
> a mean wait of about a quarter of an hour with one person and under one
> minute with two. On the same simulated days the second person saves each
> student about a quarter of an hour, and the interval of that difference is
> far from zero. Waits vary strongly between days: with one person, one
> day in four has a mean wait above 20 minutes. The result is sensitive to
> traffic. If arrivals rise by a seventh, the one-person wait grows from 16
> to 28 minutes, while the two-person wait stays near one minute.

Each sentence carries something from the list in notebook 01: the model, the
number of trials, the seed, an interval, and no more digits than the
simulation supports.

**What the model leaves out**, and which way each omission pushes:

| Left out | Effect on the real wait |
|---|---|
| Students who see a long queue and leave | shorter than simulated, and some students are not helped at all |
| A rush between classes, quiet during them | longer at the rush than any average shows |
| The desk closes; the queue at closing time | the last students wait or are sent away |
| Hard problems come in runs (an outage) | much longer on those days |
| Staff are slower when tired, faster under pressure | unknown |

The simulation answers the model, not the campus. Its use is to show which
assumptions the answer depends on, and each row of that table is a small
change to `draw_day` or `waits` away from being tested.

### The shape of a simulation study

| | This notebook | In general |
|---|---|---|
| 1 | arrivals and service are random; the rest is fixed | say what is random |
| 2 | `draw_day` | put the randomness in one function |
| 3 | `waits` | make the rest plain arithmetic |
| 4 | four days with known answers | check the arithmetic where you know the result |
| 5 | `simulate_days`, `estimate` | many trials, one generator, an interval |
| 6 | `compare_staffing` | compare options on the same draws |
| 7 | six traffic levels | vary what you guessed |
| 8 | the paragraph above | report seed, trials, interval, model |

---

## Exercises

### 🧩 Exercise 1 — Reproduce, then vary the seed

Run `compare_staffing(2000, np.random.default_rng(seed))` for five seeds of
your choice. Make a table of the mean wait with one person, with two, and the
difference. Which digits are the same in all five rows? Those are the digits
you may report.

### 🧩 Exercise 2 — How many days are enough?

Estimate the one-person mean wait from 50, 200, 800 and 3,200 days, with its
standard error each time. How many days would you need to know it to within
half a minute (an interval one minute wide)?

### 🧩 Exercise 3 — A different outcome

The dean asks: "What share of students wait more than ten minutes?" Estimate
it for one and for two staff, on the same days, with intervals. Note that the
trial is now a day, and the outcome is a share computed within the day.

### 🧩 Exercise 4 — Students who give up

Change `waits` so that a student who would have to wait more than 15 minutes
leaves without being helped (their wait counts as 15, and they take no service
time). Check your version on a case you can work out by hand first. Then
report two numbers for one staff member: the mean wait, and the share of
students who left.

### 🧩 Exercise 5 — A faster helper instead

Instead of a second person, the desk could buy software that cuts the mean
service time from 3.5 to 3.0 minutes. Compare the two remedies on the same
days. (You will need `draw_day` to take the mean service time as an argument.)
Which would you recommend, and what would you want to know about their cost?

---

**Next:** `03_Simulation_Reference` lists every tool used here, by task.
Homework 9 has the same shape as this notebook: a known answer first (the
birthday problem), then a decision with no formula (how many pastries a
coffee cart should bake).